In [ ]:
# @title Path Setup and Imports { display-mode: "form" }
# @markdown (double click to show code).

## [setup]
import math
import os
import random

import git
import magnum as mn
import numpy as np

%matplotlib inline
from matplotlib import pyplot as plt
from PIL import Image

import habitat_sim
from habitat_sim.utils import common as ut
from habitat_sim.utils import viz_utils as vut

import habitat.articulated_agents.humanoids.kinematic_humanoid as kinematic_humanoid
from habitat.articulated_agent_controllers import (
    HumanoidRearrangeController,
    HumanoidSeqPoseController,
)
from habitat.articulated_agent_controllers import Humanoid_OCRA
from omegaconf import DictConfig
from os import path as osp
import habitat_sim.agent

try:
    import ipywidgets as widgets
    from IPython.display import display as ipydisplay

    # For using jupyter/ipywidget IO components

    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

repo = git.Repo(".", search_parent_directories=True)
# dir_path = repo.working_tree_dir
dir_path = "../"
data_path = os.path.join(dir_path, "data")
output_directory = "examples/tutorials/interactivity_output/"  # @param {type:"string"}
output_path = os.path.join(dir_path, output_directory)
os.makedirs(output_path, exist_ok=True)

# define some globals the first time we run.
if "sim" not in globals():
    global sim
    sim = None
    global obj_attr_mgr
    obj_attr_mgr = None
    global prim_attr_mgr
    obj_attr_mgr = None
    global stage_attr_mgr
    stage_attr_mgr = None
    global rigid_obj_mgr
    rigid_obj_mgr = None

# @title Define Configuration Utility Functions { display-mode: "form" }
# @markdown (double click to show code)

# @markdown This cell defines a number of utility functions used throughout the tutorial to make simulator reconstruction easy:
# @markdown - make_cfg
# @markdown - make_default_settings
# @markdown - make_simulator_from_settings


def make_cfg(settings):
    sim_cfg = habitat_sim.SimulatorConfiguration()
    sim_cfg.gpu_device_id = 0
    sim_cfg.scene_id = settings["scene"]
    sim_cfg.enable_physics = settings["enable_physics"]
    if "physics_config_file" in settings:
        sim_cfg.physics_config_file = settings["physics_config_file"]
    # Optional; Specify the location of an existing scene dataset configuration
    # that describes the locations and configurations of all the assets to be used
    if "scene_dataset_config" in settings:
        sim_cfg.scene_dataset_config_file = settings["scene_dataset_config"]

    # Note: all sensors must have the same resolution
    sensor_specs = []
    if settings["color_sensor_1st_person"]:
        color_sensor_1st_person_spec = habitat_sim.CameraSensorSpec()
        color_sensor_1st_person_spec.uuid = "color_sensor_1st_person"
        color_sensor_1st_person_spec.sensor_type = habitat_sim.SensorType.COLOR
        color_sensor_1st_person_spec.resolution = [
            settings["height"],
            settings["width"],
        ]
        color_sensor_1st_person_spec.position = [0.0, settings["sensor_height"], 0.0]
        # color_sensor_1st_person_spec.orientation = [
        #     settings["sensor_pitch"],
        #     0.0,
        #     0.0,
        # ]
        color_sensor_1st_person_spec.sensor_subtype = habitat_sim.SensorSubType.PINHOLE
        sensor_specs.append(color_sensor_1st_person_spec)
    if settings["depth_sensor_1st_person"]:
        depth_sensor_1st_person_spec = habitat_sim.CameraSensorSpec()
        depth_sensor_1st_person_spec.uuid = "depth_sensor_1st_person"
        depth_sensor_1st_person_spec.sensor_type = habitat_sim.SensorType.DEPTH
        depth_sensor_1st_person_spec.resolution = [
            settings["height"],
            settings["width"],
        ]
        depth_sensor_1st_person_spec.position = [0.0, settings["sensor_height"], 0.0]
        # depth_sensor_1st_person_spec.orientation = [
        #     settings["sensor_pitch"],
        #     0.0,
        #     0.0,
        # ]
        depth_sensor_1st_person_spec.sensor_subtype = habitat_sim.SensorSubType.PINHOLE
        sensor_specs.append(depth_sensor_1st_person_spec)
    if settings["semantic_sensor_1st_person"]:
        semantic_sensor_1st_person_spec = habitat_sim.CameraSensorSpec()
        semantic_sensor_1st_person_spec.uuid = "semantic_sensor_1st_person"
        semantic_sensor_1st_person_spec.sensor_type = habitat_sim.SensorType.SEMANTIC
        semantic_sensor_1st_person_spec.resolution = [
            settings["height"],
            settings["width"],
        ]
        semantic_sensor_1st_person_spec.position = [
            0.0,
            settings["sensor_height"],
            0.0,
        ]
        # semantic_sensor_1st_person_spec.orientation = [
        #     settings["sensor_pitch"],
        #     0.0,
        #     0.0,
        # ]
        semantic_sensor_1st_person_spec.sensor_subtype = (
            habitat_sim.SensorSubType.PINHOLE
        )
        sensor_specs.append(semantic_sensor_1st_person_spec)
    if settings["color_sensor_3rd_person"]:
        color_sensor_3rd_person_spec = habitat_sim.CameraSensorSpec()
        color_sensor_3rd_person_spec.uuid = "color_sensor_3rd_person"
        color_sensor_3rd_person_spec.sensor_type = habitat_sim.SensorType.COLOR
        color_sensor_3rd_person_spec.resolution = [
            settings["height"],
            settings["width"],
        ]
        color_sensor_3rd_person_spec.position = [
            0.0,
            settings["sensor_height"] + 0.2,
            0.2,
        ]
        # color_sensor_3rd_person_spec.orientation = [-math.pi / 4, 0.0, 0.0]
        color_sensor_3rd_person_spec.sensor_subtype = habitat_sim.SensorSubType.PINHOLE
        sensor_specs.append(color_sensor_3rd_person_spec)

    # Here you can specify the amount of displacement in a forward action and the turn angle
    agent_cfg = habitat_sim.agent.AgentConfiguration()
    agent_cfg.sensor_specifications = sensor_specs
    return habitat_sim.Configuration(sim_cfg, [agent_cfg])


def make_default_settings():
    settings = {
        "width": 720,  # Spatial resolution of the observations
        "height": 544,
        "scene": os.path.join(
            data_path, "scene_datasets/mp3d_example/17DRP5sb8fy/17DRP5sb8fy.glb"
        ),  # Scene path
        "scene_dataset_config": os.path.join(
            data_path, "scene_datasets/mp3d_example/mp3d.scene_dataset_config.json"
        ),  # MP3D scene dataset
        "default_agent": 0,
        "sensor_height": 1.5,  # Height of sensors in meters
        "sensor_pitch": -math.pi / 8.0,  # sensor pitch (x rotation in rads)
        "color_sensor_1st_person": True,  # RGB sensor
        "color_sensor_3rd_person": False,  # RGB sensor 3rd person
        "depth_sensor_1st_person": False,  # Depth sensor
        "semantic_sensor_1st_person": False,  # Semantic sensor
        "seed": 1,
        "enable_physics": True,  # enable dynamics simulation
        "physics_config_file": "./data/default.physics_config.json",
    }
    return settings


In [ ]:
# @title Define Simulation Utility Functions { display-mode: "form" }
# @markdown (double click to show code)

# @markdown - remove_all_objects
# @markdown - simulate
# @markdown - sample_object_state


def simulate(sim, dt=1.0, get_frames=True):
    # simulate dt seconds at 60Hz to the nearest fixed timestep
    print("Simulating " + str(dt) + " world seconds.")
    observations = []
    start_time = sim.get_world_time()
    while sim.get_world_time() < start_time + dt:
        sim.step_physics(1.0 / 60.0)
        if get_frames:
            observations.append(sim.get_sensor_observations())
    return observations


# Set an object transform relative to the agent state
def set_object_state_from_agent(
    sim,
    obj,
    offset=np.array([0, 2.0, -1.5]),
    orientation=mn.Quaternion(((0, 0, 0), 1)),
):
    agent_transform = sim.agents[0].scene_node.transformation_matrix()
    ob_translation = agent_transform.transform_point(offset)
    obj.translation = ob_translation
    obj.rotation = orientation

def set_object_state_from_obj(
    obj_1,
    obj_2,
    offset=np.array([0, 2.0, -1.5]),
    orientation=mn.Quaternion(((0, 0, 0), 1)),
):
    ob_translation = obj_1.translation + offset
    obj_2.translation = ob_translation
    obj_2.rotation = orientation


# sample a random valid state for the object from the scene bounding box or navmesh
def sample_object_state(
    sim, obj, from_navmesh=True, maintain_object_up=True, max_tries=100, bb=None
):
    # check that the object is not STATIC
    if obj.motion_type is habitat_sim.physics.MotionType.STATIC:
        print("sample_object_state : Object is STATIC, aborting.")
    if from_navmesh:
        if not sim.pathfinder.is_loaded:
            print("sample_object_state : No pathfinder, aborting.")
            return False
    elif not bb:
        print(
            "sample_object_state : from_navmesh not specified and no bounding box provided, aborting."
        )
        return False
    tries = 0
    valid_placement = False
    # Note: following assumes sim was not reconfigured without close
    scene_collision_margin = stage_attr_mgr.get_template_by_id(0).margin
    while not valid_placement and tries < max_tries:
        tries += 1
        # initialize sample location to random point in scene bounding box
        sample_location = np.array([0, 0, 0])
        if from_navmesh:
            # query random navigable point
            sample_location = sim.pathfinder.get_random_navigable_point()
        else:
            sample_location = np.random.uniform(bb.min, bb.max)
        # set the test state
        obj.translation = sample_location
        if maintain_object_up:
            # random rotation only on the Y axis
            y_rotation = mn.Quaternion.rotation(
                mn.Rad(random.random() * 2 * math.pi), mn.Vector3(0, 1.0, 0)
            )
            obj.rotation = y_rotation * obj.rotation
        else:
            # unconstrained random rotation
            obj.rotation = ut.random_quaternion()

        # raise object such that lowest bounding box corner is above the navmesh sample point.
        if from_navmesh:
            obj_node = obj.root_scene_node
            xform_bb = habitat_sim.geo.get_transformed_bb(
                obj_node.cumulative_bb, obj_node.transformation
            )
            # also account for collision margin of the scene
            obj.translation += mn.Vector3(
                0, xform_bb.size_y() / 2.0 + scene_collision_margin, 0
            )

        # test for penetration with the environment
        if not sim.contact_test(obj.object_id):
            valid_placement = True

    if not valid_placement:
        return False
    return True

# @title Continuous Path Follower Example { display-mode: "form" }
# @markdown A python Class to provide waypoints along a path given agent states
class ContinuousPathFollowerHuman:
    def __init__(self, sim, path, human_node, waypoint_threshold):
        self._sim = sim
        self._points = path.points[:]
        assert len(self._points) > 0
        self._length = path.geodesic_distance
        self._human_node = human_node
        self._threshold = waypoint_threshold
        self._step_size = 0.01
        self.progress = 0  # geodesic distance -> [0,1]
        self.waypoint = path.points[0]

        # setup progress waypoints
        _point_progress = [0]
        _segment_tangents = []
        _length = self._length
        for ix, point in enumerate(self._points):
            if ix > 0:
                segment = point - self._points[ix - 1]
                segment_length = np.linalg.norm(segment)
                segment_tangent = segment / segment_length
                _point_progress.append(
                    segment_length / _length + _point_progress[ix - 1]
                )
                # t-1 -> t
                _segment_tangents.append(segment_tangent)
        self._point_progress = _point_progress
        self._segment_tangents = _segment_tangents
        # final tangent is duplicated
        self._segment_tangents.append(self._segment_tangents[-1])

        # print("self._length = " + str(self._length))
        # print("num points = " + str(len(self._points)))
        # print("self._point_progress = " + str(self._point_progress))
        # print("self._segment_tangents = " + str(self._segment_tangents))

    def pos_at(self, progress):
        if progress <= 0:
            return self._points[0]
        elif progress >= 1.0:
            return self._points[-1]

        path_ix = 0
        for ix, prog in enumerate(self._point_progress):
            if prog > progress:
                path_ix = ix
                break

        segment_distance = self._length * (progress - self._point_progress[path_ix - 1])
        return (
            self._points[path_ix - 1]
            + self._segment_tangents[path_ix - 1] * segment_distance
        )

    def update_waypoint(self):
        if self.progress < 1.0:
            wp_disp = self.waypoint - self._human_node.base_pos
            wp_dist = np.linalg.norm(wp_disp)
            node_pos = self._human_node.base_pos
            step_size = self._step_size
            threshold = self._threshold
            while wp_dist < threshold:
                self.progress += step_size
                self.waypoint = self.pos_at(self.progress)
                if self.progress >= 1.0:
                    break
                wp_disp = self.waypoint - node_pos
                wp_dist = np.linalg.norm(wp_disp)

class ContinuousPathFollower:
    def __init__(self, sim, path, agent_scene_node, waypoint_threshold):
        self._sim = sim
        self._points = path.points[:]
        assert len(self._points) > 0
        self._length = path.geodesic_distance
        self._node = agent_scene_node
        self._threshold = waypoint_threshold
        self._step_size = 0.01
        self.progress = 0  # geodesic distance -> [0,1]
        self.waypoint = path.points[0]

        # setup progress waypoints
        _point_progress = [0]
        _segment_tangents = []
        _length = self._length
        for ix, point in enumerate(self._points):
            if ix > 0:
                segment = point - self._points[ix - 1]
                segment_length = np.linalg.norm(segment)
                segment_tangent = segment / segment_length
                _point_progress.append(
                    segment_length / _length + _point_progress[ix - 1]
                )
                # t-1 -> t
                _segment_tangents.append(segment_tangent)
        self._point_progress = _point_progress
        self._segment_tangents = _segment_tangents
        # final tangent is duplicated
        self._segment_tangents.append(self._segment_tangents[-1])

        # print("self._length = " + str(self._length))
        # print("num points = " + str(len(self._points)))
        # print("self._point_progress = " + str(self._point_progress))
        # print("self._segment_tangents = " + str(self._segment_tangents))

    def pos_at(self, progress):
        if progress <= 0:
            return self._points[0]
        elif progress >= 1.0:
            return self._points[-1]

        path_ix = 0
        for ix, prog in enumerate(self._point_progress):
            if prog > progress:
                path_ix = ix
                break

        segment_distance = self._length * (progress - self._point_progress[path_ix - 1])
        return (
            self._points[path_ix - 1]
            + self._segment_tangents[path_ix - 1] * segment_distance
        )

    def update_waypoint(self):
        if self.progress < 1.0:
            wp_disp = self.waypoint - self._node.absolute_translation
            wp_dist = np.linalg.norm(wp_disp)
            node_pos = self._node.absolute_translation
            step_size = self._step_size
            threshold = self._threshold
            while wp_dist < threshold:
                self.progress += step_size
                self.waypoint = self.pos_at(self.progress)
                if self.progress >= 1.0:
                    break
                wp_disp = self.waypoint - node_pos
                wp_dist = np.linalg.norm(wp_disp)


def setup_path_visualization(path_follower, vis_samples=100):
    vis_objs = []
    sphere_handle = obj_attr_mgr.get_template_handles("uvSphereSolid")[0]
    sphere_template_cpy = obj_attr_mgr.get_template_by_handle(sphere_handle)
    sphere_template_cpy.scale *= 0.2
    template_id = obj_attr_mgr.register_template(sphere_template_cpy, "mini-sphere")
    print("template_id = " + str(template_id))
    if template_id < 0:
        return None
    vis_objs.append(rigid_obj_mgr.add_object_by_template_handle(sphere_handle))

    for point in path_follower._points:
        cp_obj = rigid_obj_mgr.add_object_by_template_handle(sphere_handle)
        if cp_obj.object_id < 0:
            print(cp_obj.object_id)
            return None
        cp_obj.translation = point
        vis_objs.append(cp_obj)

    for i in range(vis_samples):
        cp_obj = rigid_obj_mgr.add_object_by_template_handle("mini-sphere")
        if cp_obj.object_id < 0:
            print(cp_obj.object_id)
            return None
        cp_obj.translation = path_follower.pos_at(float(i / vis_samples))
        vis_objs.append(cp_obj)

    for obj in vis_objs:
        if obj.object_id < 0:
            print(obj.object_id)
            return None

    for obj in vis_objs:
        obj.motion_type = habitat_sim.physics.MotionType.KINEMATIC

    return vis_objs


def track_waypoint(waypoint, rs, vc, dt=1.0 / 60.0):
    angular_error_threshold = 0.5
    max_linear_speed = 1.0
    max_turn_speed = 1.0
    glob_forward = rs.rotation.transform_vector(mn.Vector3(0, 0, -1.0)).normalized()
    glob_right = rs.rotation.transform_vector(mn.Vector3(-1.0, 0, 0)).normalized()
    to_waypoint = mn.Vector3(waypoint) - rs.translation
    u_to_waypoint = to_waypoint.normalized()
    angle_error = float(mn.math.angle(glob_forward, u_to_waypoint))

    new_velocity = 0
    if angle_error < angular_error_threshold:
        # speed up to max
        new_velocity = (vc.linear_velocity[2] - max_linear_speed) / 2.0
    else:
        # slow down to 0
        new_velocity = (vc.linear_velocity[2]) / 2.0
    vc.linear_velocity = mn.Vector3(0, 0, new_velocity)

    # angular part
    rot_dir = 1.0
    if mn.math.dot(glob_right, u_to_waypoint) < 0:
        rot_dir = -1.0
    angular_correction = 0.0
    if angle_error > (max_turn_speed * 10.0 * dt):
        angular_correction = max_turn_speed
    else:
        angular_correction = angle_error / 2.0

    vc.angular_velocity = mn.Vector3(
        0, np.clip(rot_dir * angular_correction, -max_turn_speed, max_turn_speed), 0
    )

# @title Define Visualization Utility Function { display-mode: "form" }
# @markdown (double click to show code)
# @markdown - display_sample


# Change to do something like this maybe: https://stackoverflow.com/a/41432704
def display_sample(
    rgb_obs, semantic_obs=np.array([]), depth_obs=np.array([]), key_points=None
):
    from habitat_sim.utils.common import d3_40_colors_rgb

    rgb_img = Image.fromarray(rgb_obs, mode="RGBA")

    arr = [rgb_img]
    titles = ["rgb"]
    if semantic_obs.size != 0:
        semantic_img = Image.new("P", (semantic_obs.shape[1], semantic_obs.shape[0]))
        semantic_img.putpalette(d3_40_colors_rgb.flatten())
        semantic_img.putdata((semantic_obs.flatten() % 40).astype(np.uint8))
        semantic_img = semantic_img.convert("RGBA")
        arr.append(semantic_img)
        titles.append("semantic")

    if depth_obs.size != 0:
        depth_img = Image.fromarray((depth_obs / 10 * 255).astype(np.uint8), mode="L")
        arr.append(depth_img)
        titles.append("depth")

    plt.figure(figsize=(12, 8))
    for i, data in enumerate(arr):
        ax = plt.subplot(1, 3, i + 1)
        ax.axis("off")
        ax.set_title(titles[i])
        # plot points on images
        if key_points is not None:
            for point in key_points:
                plt.plot(point[0], point[1], marker="o", markersize=10, alpha=0.8)
        plt.imshow(data)

    plt.show(block=False)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--no-display", dest="display", action="store_false")
    parser.add_argument("--no-make-video", dest="make_video", action="store_false")
    parser.set_defaults(show_video=True, make_video=True)
    args, _ = parser.parse_known_args()
    show_video = args.display
    display = args.display
    make_video = args.make_video
else:
    show_video = False
    make_video = False
    display = False

In [ ]:
# @title SIM { display-mode: "form" }
# @markdown This example cell runs the object retrieval task.

# @markdown First the Simulator is re-initialized with:
# @markdown - a 3rd person camera view
# @markdown - modified 1st person sensor placement
    
sim_settings = make_default_settings()
# fmt: off
sim_settings["scene"] = "../data/scene_datasets/mp3d_example/17DRP5sb8fy/17DRP5sb8fy.glb"  # @param{type:"string"}
# fmt: on
sim_settings["sensor_pitch"] = 0
sim_settings["sensor_height"] = 0.6
sim_settings["color_sensor_3rd_person"] = True
sim_settings["depth_sensor_1st_person"] = True
sim_settings["semantic_sensor_1st_person"] = True

cfg = make_cfg(sim_settings)
# clean-up the current simulator instance if it exists
global sim
global obj_attr_mgr
global prim_attr_mgr
global stage_attr_mgr
global rigid_obj_mgr
if sim != None:
    sim.close()
# initialize the simulator
sim = habitat_sim.Simulator(cfg)

default_nav_mesh_settings = habitat_sim.NavMeshSettings()
default_nav_mesh_settings.set_defaults()
inflated_nav_mesh_settings = habitat_sim.NavMeshSettings()
inflated_nav_mesh_settings.set_defaults()
inflated_nav_mesh_settings.agent_radius = 0.2
inflated_nav_mesh_settings.agent_height = 1.5
recompute_successful = sim.recompute_navmesh(sim.pathfinder, inflated_nav_mesh_settings)
if not recompute_successful:
    print("Failed to recompute navmesh!")

# @markdown ---
# @markdown ### Set other example parameters:
seed = 100  # @param {type:"integer"}
# seed = 200  # @param {type:"integer"}
random.seed(seed)
sim.seed(seed)
np.random.seed(seed)

sim.config.sim_cfg.allow_sliding = True  # @param {type:"boolean"}

In [ ]:
# Get the bounds of scene
scene_bounds = sim.pathfinder.get_bounds()
pos = mn.Vector3(0,-3.5,0)
obj_snap = sim.pathfinder.snap_point(pos, island_index=-1)

# Get the height of scene（y axis）
print(scene_bounds)
print(obj_snap)

In [ ]:
from habitat.utils.visualizations import maps

# @markdown ###Configure Example Parameters:
# @markdown Configure the map resolution:
# meters_per_pixel = 0.08  # @param {type:"slider", min:0.01, max:1.0, step:0.01}
meters_per_pixel = 0.02  # @param {type:"slider", min:0.01, max:1.0, step:0.01}
custom_height = True  # @param {type:"boolean"}
height = 0.2  # @param {type:"slider", min:-10, max:10, step:0.1}
# display a topdown map with matplotlib
def display_map(topdown_map, key_points=None):
    plt.figure(figsize=(12, 8))
    ax = plt.subplot(1, 1, 1)
    ax.axis("off")
    plt.imshow(topdown_map)
    # plot points on map
    if key_points is not None:
        for point in key_points:
            plt.plot(point[0], point[1], marker="o", markersize=10, alpha=0.8)
    plt.show(block=False)

from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
def map2img(topdown_map, key_points=None):
    fig = plt.Figure(figsize=(12, 8), dpi=100)  
    ax = fig.add_subplot(111)  
    ax.axis("off")  
    cax = ax.imshow(topdown_map)  
    if key_points is not None:  
        for point in key_points:  
            ax.plot(point[0], point[1], marker="o", markersize=10, alpha=0.8)   
    canvas = FigureCanvas(fig)  
    canvas.draw()    
    image_data = np.frombuffer(canvas.tostring_rgb(), dtype=np.uint8)  
    image_data = image_data.reshape(fig.canvas.get_width_height()[::-1] + (3,))  
    plt.close(fig)  
    return image_data

print("The NavMesh bounds are: " + str(sim.pathfinder.get_bounds()))
if not custom_height:
    # get bounding box minumum elevation for automatic height
    height = sim.pathfinder.get_bounds()[0][1]

import math
import quaternion
def rotation2angle(quaternion_rotation):
    # Assuming rotation is the given quaternion
    rotation = quaternion.quaternion(quaternion_rotation)
    rotation.z = rotation.y
    rotation.y = 0

    yaw_angle = np.arctan2(2 * (rotation.w * rotation.z + rotation.x * rotation.y), 1 - 2 * (rotation.y**2 + rotation.z**2)) + np.pi
    # yaw_angle = np.arcsin(2 * (rotation.w * rotation.y - rotation.z * rotation.x))
    return yaw_angle

top_down_map = maps.get_topdown_map(
    sim.pathfinder, height, meters_per_pixel=meters_per_pixel
)
recolor_map = np.array(
    [[0, 255, 0], [128, 128, 128], [0, 0, 255]], dtype=np.uint8
)
top_down_map = recolor_map[top_down_map]

def show_human_obj_agent_map(human_list=[],obj=None,show=True):
    if not sim.pathfinder.is_loaded:
        print("Pathfinder not initialized, aborting.")
    else:
        # @markdown You can get the topdown map directly from the Habitat-sim API with *PathFinder.get_topdown_view*.
        # @markdown Alternatively, you can process the map using the Habitat-Lab [maps module](https://github.com/facebookresearch/habitat-api/blob/master/habitat/utils/visualizations/maps.py)
        agent_state = sim.agents[0].get_state()
        agent_pos = agent_state.position
        # print("agent_pos:",agent_pos)
        grid_dimensions = (top_down_map.shape[0], top_down_map.shape[1])
        
        agent_point = maps.to_grid(
                    agent_pos[2],
                    agent_pos[0],
                    grid_dimensions,
                    pathfinder=sim.pathfinder,
                )
        angle = rotation2angle(agent_state.rotation)
        angle = (angle-np.pi)%(2*np.pi)-np.pi
        # print("agent_state.rotation:",angle)
        # angle = -3.141592653589793/2
        top_down_map_copy = top_down_map.copy()
        maps.draw_agent(
                top_down_map_copy, agent_point, angle, agent_radius_px=8
            )
        if obj is not None:
            obj_pos = obj.translation
            # print("obj_pos:",obj_pos)
            grid_dimensions = (top_down_map.shape[0], top_down_map.shape[1])
            
            obj_point = maps.to_grid(
                        obj_pos[2],
                        obj_pos[0],
                        grid_dimensions,
                        pathfinder=sim.pathfinder,
                    )
            # angle = rotation2angle([obj.rota])
            angle = 3.141592653589793/2
            # print("obj.rotation:",angle)
            maps.draw_obj(
                    top_down_map_copy, obj_point, angle, agent_radius_px=8
                )
        
        for human in human_list:
            human_pos = human.base_pos
            # print("human_pos:",human_pos)
            grid_dimensions = (top_down_map.shape[0], top_down_map.shape[1])
            
            human_point = maps.to_grid(
                        human_pos[2],
                        human_pos[0],
                        grid_dimensions,
                        pathfinder=sim.pathfinder,
                    )
            # angle = rotation2angle(quaternion.quaternion(human.sim_obj.rotation.scalar,*human.sim_obj.rotation.vector))
            # angle = (angle+3*np.pi/2)%(2*np.pi)
            rot = mn.Quaternion.from_matrix(human.sim_obj.transformation.rotation())
            angle = rotation2angle(quaternion.quaternion(rot.scalar,*rot.vector))
            angle = (angle+np.pi)%(2*np.pi)
            angle = (angle-np.pi)%(2*np.pi)-np.pi
            # print("human angle:", angle)
            # angle = -3.141592653589793/2
            maps.draw_human(
                    top_down_map_copy, human_point, angle, agent_radius_px=8
                )
        if show:
            display_map(top_down_map_copy)
            display_sample(sim.get_sensor_observations()["color_sensor_3rd_person"])
        else:
            return map2img(top_down_map_copy)

def show_human_map(human):
    if not sim.pathfinder.is_loaded:
        print("Pathfinder not initialized, aborting.")
    else:
        # @markdown You can get the topdown map directly from the Habitat-sim API with *PathFinder.get_topdown_view*.
        # @markdown Alternatively, you can process the map using the Habitat-Lab [maps module](https://github.com/facebookresearch/habitat-api/blob/master/habitat/utils/visualizations/maps.py)
        
        human_pos = human.base_pos
        print("human_pos:",human_pos)
        grid_dimensions = (top_down_map.shape[0], top_down_map.shape[1])
        
        human_point = maps.to_grid(
                    human_pos[2],
                    human_pos[0],
                    grid_dimensions,
                    pathfinder=sim.pathfinder,
                )
        angle = rotation2angle(human.base_rot)
        # print("human.base_rot:", human.base_rot)
        # angle = -3.141592653589793/2
        top_down_map_copy = top_down_map.copy()
        maps.draw_human(
                top_down_map_copy, human_point, human.base_rot, agent_radius_px=8
            )
        
        display_map(top_down_map_copy)

def show_agent_map():
    if not sim.pathfinder.is_loaded:
        print("Pathfinder not initialized, aborting.")
    else:
        # @markdown You can get the topdown map directly from the Habitat-sim API with *PathFinder.get_topdown_view*.
        # @markdown Alternatively, you can process the map using the Habitat-Lab [maps module](https://github.com/facebookresearch/habitat-api/blob/master/habitat/utils/visualizations/maps.py)
        
        agent_state = sim.agents[0].get_state()
        agent_pos = agent_state.position
        # print("agent_pos:",agent_pos)
        grid_dimensions = (top_down_map.shape[0], top_down_map.shape[1])
        
        agent_point = maps.to_grid(
                    agent_pos[2],
                    agent_pos[0],
                    grid_dimensions,
                    pathfinder=sim.pathfinder,
                )
        angle = rotation2angle(agent_state.rotation)
        print("angle:",angle)
        # angle = agent_state.rotation.angle()
        # print("agent_state.rotation:",agent_state.rotation)
        # angle = -3.141592653589793/2
        top_down_map_copy = top_down_map.copy()
        maps.draw_agent(
                top_down_map_copy, agent_point, angle, agent_radius_px=8
            )
        
        display_map(top_down_map_copy)
        display_sample(sim.get_sensor_observations()["color_sensor_3rd_person"])


In [ ]:
def make_human(humanoid_name,sim,base_pos=None):
    humanoid_path = f"../data/humanoids/humanoid_data/{humanoid_name}/{humanoid_name}.urdf"
    walk_pose_path = f"../data/humanoids/humanoid_data/{humanoid_name}/{humanoid_name}_motion_data_smplx.pkl"
    agent_config = DictConfig(
        {
            "articulated_agent_urdf": humanoid_path,
            "motion_data_path": walk_pose_path,
        }
    )
    if not osp.exists(humanoid_path):
        print(f"No humanoid file {humanoid_path}")
    kin_humanoid = kinematic_humanoid.KinematicHumanoid(agent_config, sim)
    kin_humanoid.reconfigure()
    kin_humanoid.update()
    if base_pos is not None:
        kin_humanoid.base_pos = base_pos
    else:
        while 1:
            base_pos = sim.pathfinder.get_random_navigable_point()
            base_pos = sim.pathfinder.snap_point(
                    base_pos
                )
            base_pos[1] = 0.05
            if sim.pathfinder.distance_to_closest_obstacle(base_pos) > 0.1:
                kin_humanoid.base_pos = base_pos
                break
    return kin_humanoid,agent_config

In [ ]:
# add the humanoid to the world via the wrapper
human_list = []
human_config_list = []
type_list = ['male_0','female_1','male_2']
for huamn_type in type_list:
    kin_humanoid,human_config = make_human(huamn_type,sim)
    human_list.append(kin_humanoid)
    human_config_list.append(human_config)

# set base ground position from navmesh
# NOTE: because the navmesh floats above the collision geometry we should see a pop/settle with dynamics and no fixed base
# target_base_pos = sim.pathfinder.snap_point(
#     kin_humanoid.sim_obj.translation
# )


# Managers of various Attributes templates
obj_attr_mgr = sim.get_object_template_manager()
obj_attr_mgr.load_configs(str(os.path.join(data_path, "objects/example_objects")))
obj_attr_mgr.load_configs(str(os.path.join(data_path, "objects/locobot_merged")))
prim_attr_mgr = sim.get_asset_template_manager()
stage_attr_mgr = sim.get_stage_template_manager()
# Manager providing access to rigid objects
rigid_obj_mgr = sim.get_rigid_object_manager()
# rigid_obj_mgr.remove_all_objects()

# load a selected target object and place it on the NavMesh
cheezit_handle = obj_attr_mgr.get_template_handles("cheezit")[0]
obj_1 = rigid_obj_mgr.add_object_by_template_handle(cheezit_handle)
base_pos = sim.pathfinder.snap_point(
        sim.pathfinder.get_random_navigable_point()
    )
base_pos[0] = 2.5
base_pos[1] = 0.1
base_pos[2] = -3.2
obj_1.translation = base_pos
print("base_pos:",base_pos)
# obj_1.translation = [3, 0.5, 6]

# load the locobot_merged asset
locobot_template_handle = obj_attr_mgr.get_file_template_handles("locobot")[0]

# add robot object to the scene with the agent/camera SceneNode attached
locobot_obj = rigid_obj_mgr.add_object_by_template_handle(
    locobot_template_handle, sim.agents[0].scene_node
)
# sim.agents[0].scene_node.translation = [38.0 , 0.1  , 13.0]
# sim.agents[0].scene_node.translation = [15.0 , 0.1  , -6.0]
sim.agents[0].scene_node.translation = [-5.2, 0.2, 0.8]

# base_pos = sim.pathfinder.snap_point(
#         sim.pathfinder.get_random_navigable_point()
#     )
# base_pos[1] = 0.1
# sim.agents[0].scene_node.translation = base_pos
print("sim.agents[0].scene_node.translation:",sim.agents[0].scene_node.translation)
# set the agent's body to kinematic since we will be updating position manually
locobot_obj.motion_type = habitat_sim.physics.MotionType.KINEMATIC

In [ ]:
display_sample(sim.get_sensor_observations()["color_sensor_3rd_person"])

In [ ]:
# obtain the default, discrete actions that an agent can perform
# default action space contains 3 actions: move_forward, turn_left, and turn_right
# 定义了包含三个动作的离散动作空间：前进、左转和右转，可以自定义离散动作空间自定义动作
action_names = list(cfg.agents[sim_settings["default_agent"]].action_space.keys())
print("Discrete action space: ", action_names)

import quaternion
def rotation2angle(quaternion_rotation):
    # Assuming rotation is the given quaternion
    rotation = quaternion.quaternion(quaternion_rotation)
    rotation.z = rotation.y
    rotation.y = 0

    yaw_angle = np.arctan2(2 * (rotation.w * rotation.z + rotation.x * rotation.y), 1 - 2 * (rotation.y**2 + rotation.z**2)) + np.pi
    # yaw_angle = np.arcsin(2 * (rotation.w * rotation.y - rotation.z * rotation.x))
    return yaw_angle

def get_robot_state(robot):
    px,_,py = robot.translation
    theta = rotation2angle(quaternion.quaternion(robot.rotation.scalar,*robot.rotation.vector))
    print([px,py,theta])

def navigateAndSee(action=""):
    if action in action_names:
        observations = sim.step(action)
        print("action: ", action)
        if display:
            # display_sample(sim.get_sensor_observations()["color_sensor_3rd_person"])
            # show_agent_map()
            get_robot_state(locobot_obj)

In [ ]:
# show_agent_map()
# show_human_map(human=kin_humanoid)
show_human_obj_agent_map(human_list=human_list,obj=obj_1)

In [ ]:

import cv2
import numpy as np

map_array = maps.get_topdown_map(
    sim.pathfinder, 0.2, meters_per_pixel=0.01
)

boder = sim.pathfinder.get_bounds()
boder = [boder[0][0],boder[0][2],boder[1][0],boder[1][2]]

contours, hierarchy = cv2.findContours(map_array.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

outer_boundary = None
inner_boundaries = []

for i, contour in enumerate(contours):
    if hierarchy[0][i][3] == -1: 
        outer_boundary = contour.squeeze()
    else: 
        inner_boundaries.append(contour.squeeze())

p1 = outer_boundary[outer_boundary[:, 0].argmin()]
p2 = outer_boundary[outer_boundary[:, 0].argmax()]

idx_p1 = np.where((outer_boundary[:, 0] == p1[0]) & (outer_boundary[:, 1] == p1[1]))[0][0]
idx_p2 = np.where((outer_boundary[:, 0] == p2[0]) & (outer_boundary[:, 1] == p2[1]))[0][0]

if idx_p1 < idx_p2:
    new_outer_boundary1 = np.concatenate((outer_boundary[idx_p1:idx_p2+1], 
        [[map_array.shape[1], map_array.shape[0]], [0, map_array.shape[0]], [0, 0], [p1[0],p1[1]]]))
    new_outer_boundary2 = np.concatenate((outer_boundary[idx_p2:],outer_boundary[:idx_p1+1], 
        [[0, 0], [map_array.shape[1], 0], [map_array.shape[1], map_array.shape[0]], [p2[0],p2[1]]]))
else:
    new_outer_boundary1 = np.concatenate((outer_boundary[idx_p2:idx_p1+1], 
        [[map_array.shape[1], map_array.shape[0]], [0, map_array.shape[0]], [0, 0], [p2[0],p2[1]]]))
    new_outer_boundary2 = np.concatenate((outer_boundary[idx_p1:],outer_boundary[:idx_p2+1],  
        [[0, 0], [map_array.shape[1], 0], [map_array.shape[1], map_array.shape[0]], [p1[0],p1[1]]]))

map_with_obstacles = np.ones(map_array.shape, dtype=np.uint8) * 255  

inner_boundaries.append(new_outer_boundary1.reshape((-1,  2)).astype(np.int32))
inner_boundaries.append(new_outer_boundary2.reshape((-1, 2)).astype(np.int32))
for boundary in inner_boundaries:
    cv2.fillPoly(map_with_obstacles, [boundary], color=0)

obstacles = []
for boundary in inner_boundaries:
    real_boundary = []
    for i in range(boundary.shape[0]):
        x_real = boder[0] + (boundary[i][0] / map_array.shape[1]) * (boder[2] - boder[0])
        y_real = boder[1] + (boundary[i][1] / map_array.shape[0]) * (boder[3] - boder[1])
        real_boundary.append([x_real, y_real])
    obstacles.append(real_boundary)
cv2.imwrite("Map.png", map_with_obstacles)


In [ ]:
from collections import deque

class HumanPoseContrioller:
    def __init__(self,sim,human,motion_path,direction=mn.Vector3(0,0,1)):
        self.walk_flag = False
        motion_list = [
            "../data/humanoids/action/79_70_stageii.pkl",  # 	laughing
            "../data/humanoids/action/13_23_stageii.pkl",  # 	sweep floor
            # "../data/humanoids/action/05_02_stageii.pkl"    # dance
        ]
        motion_path = random.choice(motion_list)
        self.human = human
        self.human.walk_flag = self.walk_flag
        self.sim = sim
        # We define here humanoid controller
        self.humanoid_controller = HumanoidSeqPoseController(motion_path)
        base_trans = self.human.base_transformation
        self.humanoid_controller.reset(base_trans)
        self.humanoid_controller.set_direction(direction) # 初始朝向，决定动作方向
    
    def move(self):
        self.humanoid_controller.calculate_pose()
        self.humanoid_controller.next_pose(True)

        # The get_pose function gives as a humanoid pose in the same format as HumanoidJointAction
        new_pose = self.humanoid_controller.get_pose()

        new_joints = new_pose[:-16]
        new_pos_transform_base = new_pose[-16:]
        new_pos_transform_offset = new_pose[-32:-16]

        # When the array is all 0, this indicates we are not setting
        # the human joint
        if np.array(new_pos_transform_offset).sum() != 0:
            vecs_base = [
                mn.Vector4(new_pos_transform_base[i * 4 : (i + 1) * 4])
                for i in range(4)
            ]
            vecs_offset = [
                mn.Vector4(new_pos_transform_offset[i * 4 : (i + 1) * 4])
                for i in range(4)
            ]
            new_transform_offset = mn.Matrix4(*vecs_offset)
            new_transform_base = mn.Matrix4(*vecs_base)
            self.human.set_joint_transform(
                new_joints, new_transform_offset, new_transform_base
            )

class HumanWalkController:
    def __init__(self,sim,human,human_config) -> None:
        self.walk_flag = True
        self.path = habitat_sim.ShortestPath()
        self.sim = sim
        self.human = human
        self.human.walk_flag = self.walk_flag
        self.humanoid_controller = HumanoidRearrangeController(human_config["motion_data_path"])
        self.humanoid_controller.reset(self.human.base_transformation)
        self.continuous_path_follower = None
        self.last_process = -1
        self.queue_len = 5
        
        self.queue = deque(range(-1, -1-self.queue_len))
        # self.get_random_path()
        self.set_gxy()
    
    # 函数用于添加新数据并判断队列中的元素是否相同
    def update_queue(self,data):
        # 将新数据添加到队列的右侧
        self.queue.append(data)
        
        # 如果队列长度超过self.queue_len，从左侧移除一个元素
        if len(self.queue) > self.queue_len :
            self.queue.popleft()
        
        # 检查队列中的所有元素是否相同
        all_equal = all(x == self.queue[0] for x in self.queue)
        return all_equal
    
    def get_random_path(self):
        found_path = False
        
        while not found_path:
            self.path.requested_start = self.human.base_pos
            requested_end = self.sim.pathfinder.get_random_navigable_point()
            requested_end = sim.pathfinder.snap_point(
                requested_end
            )
            requested_end[1] = 0.7
            self.path.requested_end = requested_end
            # self.path.requested_end = self.human.base_pos + mn.Vector3(0.0, 0.0, 10.51752)
            print("self.path.requested_end:",self.path.requested_end)

            found_path = self.sim.pathfinder.find_path(self.path)
        self.continuous_path_follower = ContinuousPathFollowerHuman(
            self.sim, self.path, self.human, waypoint_threshold=0.4
        )
    
    def human_move_vxy(self,vx,vy):
        self.humanoid_controller.calculate_walk_pose_vxy(vx=vx,vy=vy)
        new_pose = self.humanoid_controller.get_pose()
        self.human.set_vxy(self.humanoid_controller.vx,self.humanoid_controller.vy)

        new_joints = new_pose[:-16]
        new_pos_transform_base = new_pose[-16:]
        new_pos_transform_offset = new_pose[-32:-16]

        # When the array is all 0, this indicates we are not setting
        # the human joint
        if np.array(new_pos_transform_offset).sum() != 0:
            vecs_base = [
                mn.Vector4(new_pos_transform_base[i * 4 : (i + 1) * 4])
                for i in range(4)
            ]
            vecs_offset = [
                mn.Vector4(new_pos_transform_offset[i * 4 : (i + 1) * 4])
                for i in range(4)
            ]
            new_transform_offset = mn.Matrix4(*vecs_offset)
            new_transform_base = mn.Matrix4(*vecs_base)
            self.human.set_joint_transform(
                new_joints, new_transform_offset, new_transform_base
            )
        # print("state:",self.human.get_state())
        
    def human_move_diff(self,pose_diff):
        self.humanoid_controller.calculate_walk_pose(pose_diff)
        new_pose = self.humanoid_controller.get_pose()

        new_joints = new_pose[:-16]
        new_pos_transform_base = new_pose[-16:]
        new_pos_transform_offset = new_pose[-32:-16]

        # When the array is all 0, this indicates we are not setting
        # the human joint
        if np.array(new_pos_transform_offset).sum() != 0:
            vecs_base = [
                mn.Vector4(new_pos_transform_base[i * 4 : (i + 1) * 4])
                for i in range(4)
            ]
            vecs_offset = [
                mn.Vector4(new_pos_transform_offset[i * 4 : (i + 1) * 4])
                for i in range(4)
            ]
            new_transform_offset = mn.Matrix4(*vecs_offset)
            new_transform_base = mn.Matrix4(*vecs_base)
            self.human.set_joint_transform(
                new_joints, new_transform_offset, new_transform_base
            )
        print("state:",self.human.get_state())
    
    def human_move(self,target_pos):
        pose_diff = target_pos - self.human.base_pos
        self.humanoid_controller.calculate_walk_pose(pose_diff)
        new_pose = self.humanoid_controller.get_pose()

        new_joints = new_pose[:-16]
        new_pos_transform_base = new_pose[-16:]
        new_pos_transform_offset = new_pose[-32:-16]

        # When the array is all 0, this indicates we are not setting
        # the human joint
        if np.array(new_pos_transform_offset).sum() != 0:
            vecs_base = [
                mn.Vector4(new_pos_transform_base[i * 4 : (i + 1) * 4])
                for i in range(4)
            ]
            vecs_offset = [
                mn.Vector4(new_pos_transform_offset[i * 4 : (i + 1) * 4])
                for i in range(4)
            ]
            new_transform_offset = mn.Matrix4(*vecs_offset)
            new_transform_base = mn.Matrix4(*vecs_base)
            self.human.set_joint_transform(
                new_joints, new_transform_offset, new_transform_base
            )
    
    def set_gxy(self):
        found_path = False
        
        while not found_path:
            self.path.requested_start = self.human.base_pos
            requested_end = self.sim.pathfinder.get_random_navigable_point()
            requested_end = sim.pathfinder.snap_point(
                requested_end
            )
            requested_end[1] = 0.7
            self.path.requested_end = requested_end
            # self.path.requested_end = self.human.base_pos + mn.Vector3(0.0, 0.0, 10.51752)

            found_path = self.sim.pathfinder.find_path(self.path)  
        print("requested_end:",requested_end)
        self.human.gx = requested_end[0]
        self.human.gy = requested_end[2]
        
        # return requested_end[0], requested_end[2]
    
    def run_follow_orca(self):
        distance = (self.human.gx - self.human.px)**2 + (self.human.gy - self.human.gx)**2
        distance = round(distance, 2)
        collided = self.update_queue(distance)
        if collided or distance < 1:
            print("reset gxy")
            self.set_gxy()
    
    def run_follow_waypoint(self):
        if (self.continuous_path_follower.progress > 0.75):
            self.get_random_path()
        # if self.last_process == self.continuous_path_follower.progress:  # reset
            
        self.last_process = self.continuous_path_follower.progress
        self.continuous_path_follower.update_waypoint()
        end_pos = sim.step_filter(
            self.human.base_pos, mn.Vector3(self.continuous_path_follower.waypoint)
        )
        # # Check if a collision occured
        # dist_moved_before_filter = (
        #     mn.Vector3(self.continuous_path_follower.waypoint) - self.human.base_pos
        # ).dot()
        # dist_moved_after_filter = (end_pos - self.human.base_pos).dot()

        # # NB: There are some cases where ||filter_end - end_pos|| > 0 when a
        # # collision _didn't_ happen. One such case is going up stairs.  Instead,
        # # we check to see if the the amount moved after the application of the filter
        # # is _less_ than the amount moved before the application of the filter
        # EPS = 1e-5
        # collided = (dist_moved_after_filter + EPS) < dist_moved_before_filter
        collided = self.update_queue(self.continuous_path_follower.progress)
        # print("collided:",collided)
        if collided:
            self.get_random_path()
        else:
            self.human_move(end_pos)

In [ ]:
# create and configure a new VelocityControl structure
# Note: this is NOT the object's VelocityControl, so it will not be consumed automatically in sim.step_physics
vel_control = habitat_sim.physics.VelocityControl()
vel_control.controlling_lin_vel = True
vel_control.lin_vel_is_local = True
vel_control.controlling_ang_vel = True
vel_control.ang_vel_is_local = True


# human controller
Human_ocra = Humanoid_OCRA(obstacles)
HumanControl_list = []
for i in range(len(human_list)):
    num = random.random()
    if num < 0.3:
        HumanControl = HumanWalkController(sim,human_list[i],human_config_list[i])
    else:
        HumanControl = HumanPoseContrioller(sim,human_list[i],human_config_list[i])
    HumanControl_list.append(HumanControl)

# reset observations and robot state
# locobot_obj.translation = sim.pathfinder.get_random_navigable_point()
observations = []
map_imgs = []

# get shortest path to the object from the agent position
found_path = False
path1 = habitat_sim.ShortestPath()
path2 = habitat_sim.ShortestPath()
while not found_path:
    # if not sample_object_state(
    #     sim, obj_1, from_navmesh=True, maintain_object_up=True, max_tries=1000
    # ):
    #     print("Couldn't find an initial object placement. Aborting.")
    #     break
    path1.requested_start = locobot_obj.translation
    path1.requested_end = obj_1.translation
    path2.requested_start = path1.requested_end
    path2.requested_end = sim.pathfinder.get_random_navigable_point()

    found_path = sim.pathfinder.find_path(path1) and sim.pathfinder.find_path(path2)

if not found_path:
    print("Could not find path to object, aborting!")

vis_objs = []

recompute_successful = sim.recompute_navmesh(sim.pathfinder, default_nav_mesh_settings)
if not recompute_successful:
    print("Failed to recompute navmesh 2!")


continuous_path_follower = ContinuousPathFollower(
    sim, path1, locobot_obj.root_scene_node, waypoint_threshold=0.4
)

show_waypoint_indicators = True  # @param {type:"boolean"}
time_step = 1.0 / 30.0
for i in range(1):
    # if i == 1:
    #     continuous_path_follower = ContinuousPathFollower(
    #         sim, path2, locobot_obj.root_scene_node, waypoint_threshold=0.4
    #     )

    if show_waypoint_indicators:
        for vis_obj in vis_objs:
            rigid_obj_mgr.remove_object_by_id(vis_obj.object_id)
        vis_objs = setup_path_visualization(continuous_path_follower)

    # manually control the object's kinematic state via velocity integration
    start_time = sim.get_world_time()
    max_time = 20
    while (
        continuous_path_follower.progress < 1.0
        and sim.get_world_time() - start_time < max_time
    ):
        # human
        # for hc in HumanControl_list:
        #     hc.run_follow_waypoint()
        for _ in range(3):
            human_actions = Human_ocra.get_huamn_actions(human_list)
            for i,action in enumerate(human_actions):
                hc = HumanControl_list[i]
                if hc.walk_flag:
                    hc.human_move_vxy(vx=action.vx,vy=action.vy)
                    hc.run_follow_orca()
                else:
                    hc.move()

        continuous_path_follower.update_waypoint()
        if show_waypoint_indicators:
            vis_objs[0].translation = continuous_path_follower.waypoint

        if locobot_obj.object_id < 0:
            print("locobot_id " + str(locobot_obj.object_id))
            break

        previous_rigid_state = locobot_obj.rigid_state

        # set velocities based on relative waypoint position/direction
        track_waypoint(
            continuous_path_follower.waypoint,
            previous_rigid_state,
            vel_control,
            dt=time_step,
        )

        # manually integrate the rigid state
        target_rigid_state = vel_control.integrate_transform(
            time_step, previous_rigid_state
        )

        # snap rigid state to navmesh and set state to object/agent
        end_pos = sim.step_filter(
            previous_rigid_state.translation, target_rigid_state.translation
        )
        locobot_obj.translation = end_pos
        locobot_obj.rotation = target_rigid_state.rotation

        # Check if a collision occured
        dist_moved_before_filter = (
            target_rigid_state.translation - previous_rigid_state.translation
        ).dot()
        dist_moved_after_filter = (end_pos - previous_rigid_state.translation).dot()

        # NB: There are some cases where ||filter_end - end_pos|| > 0 when a
        # collision _didn't_ happen. One such case is going up stairs.  Instead,
        # we check to see if the the amount moved after the application of the filter
        # is _less_ than the amount moved before the application of the filter
        EPS = 1e-5
        collided = (dist_moved_after_filter + EPS) < dist_moved_before_filter

        # run any dynamics simulation
        sim.step_physics(time_step)

        # render observation
        observations.append(sim.get_sensor_observations())
        map_img = show_human_obj_agent_map(human_list=human_list,obj=obj_1,show=False)
        map_imgs.append(map_img)
        
        # observations += simulate(sim, 0.01, True)
        # display_sample(sim.get_sensor_observations()["color_sensor_1st_person"])

# release
# gripper.release()
# start_time = sim.get_world_time()
# while sim.get_world_time() - start_time < 2.0:
#     sim.step_physics(time_step)
#     observations.append(sim.get_sensor_observations())

# video rendering with embedded 1st person view
video_prefix = "fetch"
if make_video:
    overlay_dims = (int(sim_settings["width"] / 5), int(sim_settings["height"] / 5))
    print("overlay_dims = " + str(overlay_dims))
    overlay_settings = [
        {
            "obs": "depth_sensor_1st_person",
            "type": "depth",
            "dims": overlay_dims,
            "pos": (10, 10),
            "border": 2,
        },
        # {
        #     "obs": "color_sensor_1st_person",
        #     "type": "color",
        #     "dims": overlay_dims,
        #     "pos": (10, 10),
        #     "border": 2,
        # },
        # {
        #     "obs": "depth_sensor_1st_person",
        #     "type": "depth",
        #     "dims": overlay_dims,
        #     "pos": (10, 30 + overlay_dims[1]),
        #     "border": 2,
        # },
    ]
    print("overlay_settings = " + str(overlay_settings))

    vut.make_video(
        observations=observations,
        primary_obs="color_sensor_3rd_person",
        primary_obs_type="color",
        video_file=output_path + video_prefix,
        fps=int(1.0 / time_step),
        open_vid=show_video,
        overlay_settings=overlay_settings,
        depth_clip=10.0,
    )
      
video_filename = 'map_video_indoor.mp4'  
import imageio  
imageio.mimsave(video_filename, map_imgs, fps=int(1.0 / time_step))

# remove locobot while leaving the agent node for later use
rigid_obj_mgr.remove_object_by_id(locobot_obj.object_id, delete_object_node=False)
rigid_obj_mgr.remove_all_objects()